# Sacks 2026: CNV dose response of height and BMI - All of Us Researcher Workbench
Same code base as the UK Biobank analysis (branch `cnv-gwas` of jsebat/cnv-growth-bmi, folder `AllOfUs/`); only the config differs. Controlled Tier, Python kernel. Read `AllOfUs/HANDOFF_Milad.md` first.

Federated design: individual-level files stay on this disk and in the workspace bucket. Only `export_aggregate/` (aggregate tables + null-replicate test statistics, after the small-cell rule) leaves; it goes to UCSD (Expanse), where the meta-analysis with UK Biobank is run. Clear large cell outputs before saving the notebook (egress alerts).

In [ ]:
import os, subprocess, json
print(os.environ.get('WORKSPACE_CDR'), os.environ.get('WORKSPACE_BUCKET'), os.environ.get('GOOGLE_PROJECT'))
ROOT = os.path.expanduser('~/cnv-growth-bmi/AllOfUs')
if not os.path.exists(ROOT):
    subprocess.run(['git', 'clone', '-b', 'cnv-gwas', 'https://github.com/jsebat/cnv-growth-bmi.git', os.path.expanduser('~/cnv-growth-bmi')], check=True)
CODE = f'{ROOT}/cnv_anthro'; CFG = 'configs/aou_sacks2026.json'
for d in ['data', 'results']:
    os.makedirs(os.path.expanduser(f'~/cnv_anthro_run/{d}'), exist_ok=True)

## 0. Resources bundle
Upload `cnv_resources_sacks2026_2026-11-16.tar.gz` (built on Expanse by `ukbb/make_aou_bundle_sacks2026.sh`; check its `.sha256`) to the workspace bucket. It holds the public annotations and the verified UKBB shift files (`resources/shared_shifts/`).

In [ ]:
BUNDLE = 'cnv_resources_sacks2026_2026-11-16.tar.gz'
if not os.path.exists(f'{ROOT}/resources'):
    subprocess.run(f"gsutil cp $WORKSPACE_BUCKET/{BUNDLE} {ROOT}/ && cd {ROOT} && sha256sum {BUNDLE} && tar xzf {BUNDLE}", shell=True, check=True)
print(sorted(os.listdir(f'{ROOT}/resources'))); print(open(f'{ROOT}/resources/shared_shifts/SHIFTS_MANIFEST.tsv').read())

## 1. Fill the two data paths
From the Workbench page on how the genomic data are organised (version matching this workspace's CDR): the srWGS **GATK-SV joint-called VCF** and **ancestry_preds.tsv**.

In [ ]:
cfg_path = f'{CODE}/{CFG}'
cfg = json.load(open(cfg_path))
SV_VCF = 'gs://...'          # <- fill
ANCESTRY_GS = 'gs://...'     # <- fill
anc = os.path.expanduser('~/cnv_anthro_run/data/ancestry_preds.tsv')
subprocess.run(['gsutil', '-u', os.environ['GOOGLE_PROJECT'], 'cp', ANCESTRY_GS, anc], check=True)
cfg['prep_commands'][0][2] = SV_VCF
cfg['prep_commands'][1][cfg['prep_commands'][1].index('--ancestry') + 1] = anc
json.dump(cfg, open(cfg_path, 'w'), indent=1)

## 2. Prep (SV VCF -> CNV table; OMOP -> phenotype table)

In [ ]:
!cd {CODE} && python run_pipeline.py --config {CFG} --steps prep
print(open(os.path.expanduser('~/cnv_anthro_run/results/aou_pheno_summary.json')).read())

## 3. CNV GWAS, S9 alleles, growth-locus effects, genome-wide burden - every stratum

In [ ]:
!cd {CODE} && python run_pipeline.py --config {CFG} --steps gwas,plots,alleles,loci,burden

## 4. Figure analyses in the stratum `all`: additivity, axis scores, univariate grid, call size, joint models
The grid and joint steps are the long ones; run them from a terminal with `nohup` (see the handoff).

In [ ]:
!cd {CODE} && python run_pipeline.py --config {CFG} --steps additivity,scores --strata all

In [ ]:
# long: run in a terminal instead
# cd ~/cnv-growth-bmi/AllOfUs/cnv_anthro && nohup python run_pipeline.py --config configs/aou_sacks2026.json --steps grid,joint --strata all --jobs 24 > ~/cnv_anthro_run/grid_joint.log 2>&1 &

## 5. Export (small-cell rule + cross-stratum suppression)
Check `EXPORT_MANIFEST.txt` (size, withheld rows) before any download.

In [ ]:
!cd {CODE} && python run_pipeline.py --config {CFG} --steps export
print(open(os.path.expanduser('~/cnv_anthro_run/export_aggregate/EXPORT_MANIFEST.txt')).read()[:800])

## 6. Package the summary statistics for UCSD
The meta-analysis runs on Expanse (`ukbb/meta_sacks2026_expanse.sbatch`), not here. Package `export_aggregate/` with its `null_runs.npz` files (test statistics only; the meta-analysis needs them), check the size against the egress limit, download via the file browser, and `scp` to `/expanse/projects/sebat1/jsebat/sacks2026_aou/aou_results/`.

In [ ]:
!cd ~/cnv_anthro_run && T=aou_sacks2026_export_$(date +%Y%m%d).tar.gz && tar czf $T export_aggregate && sha256sum $T > $T.sha256 && ls -la $T*